In [1]:
# imports
import torch
from models.bert import ContinuityBERT
from train import parse_args
import create_knowledge_graph as kg_utils
from data import utils

In [2]:
# Load the right model that we want to analyze
def create_bert_model(config):
    encoder_type = config["encoder_type"]
    use_kg = "kg" in config["model_type"]
    model = ContinuityBERT(
        n_heads=config["n_heads"],
        n_layers=config["n_layers"],
        n_gnn_layers=config["n_gnn_layers"],
        hidden_dim=config["hidden_dim"],
        input_dim=utils.SENTENCE_ENCODER_DIM[encoder_type],
        use_kg=use_kg,
        kg_node_dim=kg_utils.KG_NODE_DIM,
        kg_edge_dim=kg_utils.KG_EDGE_DIM,
        dropout=config["dropout"],
        gnn_type=config["gnn_type"],
    )
    return model

config_bert_kg_gat = {
    "train_ratio": 0.5,
    "batch_size": 64,
    "n_continuity_errors": 1, #[1, 2
    "n_heads": 8,
    "n_layers": 3,
    "n_gnn_layers": 2,
    "hidden_dim": 20,
    "dropout": 0.2,
    "n_epochs": 100,
    "n_runs": 5,
    "lr": 1e-5,
    "pr_threshold": 0.3,
    "encoder_type": "all-MiniLM-L6-v2",
    "gnn_type": "gatv2", #["gatv2", "gcn"],
    "model_type": "bert_kg"
}

model = create_bert_model(config_bert_kg_gat)
model.eval()

initialized continuityBERT with 628261 parameters.


ContinuityBERT(
  (embedder): Linear(in_features=384, out_features=160, bias=True)
  (decider): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=160, out_features=160, bias=True)
        )
        (linear1): Linear(in_features=160, out_features=160, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=160, out_features=160, bias=True)
        (norm1): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (gats): ModuleList(
    (0-1): 2 x GATv2Conv(100, 100, heads=1)
  )
  (aggregator): MeanAggregation()
  (proj): Linear(in_features=260, out_features=1, bias=True)
  (softmax): Softmax(dim=-1)
)

In [3]:
# Load saved weights into 
MODEL_WEIGHTS_PATH = "./results/bert_kg_gat/bert_kg_gat-1_error-params.pkl"
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH))

<All keys matched successfully>

In [4]:
# Load example data
import pickle as pkl
input_data = "./data/dataset/encoded/test/test_1_error.pkl"
with open(input_data, "rb") as f:
    dataset, _ = pkl.load(f)

xs, ys = [], []
kgs = []
docs = []
for i, (x, y, kg, doc) in enumerate(dataset):
    xs.append(x)
    ys.append(y)
    kgs.append(kg)
    docs.append(doc)

In [15]:
# 5, 8, 24
# example_datapoint = 5 # example 1
# promising -- 173
# meh but ? -- 66? 80? 223


# FINAL: 194, 204(?), 360

found = False
example_datapoint = 321

while not found:
    matched = False

    while not matched:
        #torch.manual_seed(0)
        x = xs[example_datapoint]
        y = ys[example_datapoint]
        kg = kgs[example_datapoint]

        # datapoint 5 == 1_error/test/synthetic_kaggle_2000_776_continuity7.txt
        # datapoint 8 == .../synthetic_kaggle_2254_779_continuity4.txt
        import os
        osl = os.listdir
        ospj = os.path.join
        orig_docs_path = "data/dataset/1_error/test/"
        def find_original_doc(kg, y, kgidx=0):
            node_labels = kg["node_labels"]
            if len(node_labels) <= kgidx: return [(None, None)]
            ds = [x for x in osl(orig_docs_path) if x.endswith(".txt")]
            matches = []
            for d in ds:
                with open(ospj(orig_docs_path, d)) as f:
                    lines = f.readlines()
                txtmatch = node_labels[kgidx]
                max_idx = torch.argmax(y).item()
                #if d == "synthetic_kaggle_2701_902_continuity6.txt":
                #    print(max_idx, lines[0])
                #    print(str(max_idx) in lines[0])
                #    print(txtmatch in " ".join(lines))
                #    print(txtmatch)
                if not (f"[{max_idx}]" in lines[0] and txtmatch in " ".join(lines)):
                    continue
                matches.append((d, lines))
            if not matches:
                #print(f"WARNING: no match found for kgidx={kgidx}!")
                return find_original_doc(kg, y, kgidx=kgidx+1)
                matches = [(None, None)]
            #print(f"INFO: {len(matches)} matching doc(s) found at kgidx={kgidx}")

            # filter matches if more than 1
            kgi = kgidx
            while len(matches) > 1:
                new_matches = []
                kgi += 1
                #print(f"DEBUG: filtering with kgi={kgi}")
                if kgi >= len(node_labels):
                    print(f"WARNING: unfilterable down to 1 orig data doc! {len(matches)} matches remaining")
                    break
                txtmatch = node_labels[kgi]
                for d, lines in matches:
                    if not txtmatch in " ".join(lines):
                        continue
                    new_matches.append((d, lines))
                if new_matches:
                    matches = new_matches
            #if len(matches) == 1: print(f"INFO: sufficiently filtered at kgidx={kgi}")

            return matches#[0]
        #print(f"data_{example_datapoint} ")
        orig_data_file, orig_data_lines = find_original_doc(kg, y)[0]
        orig_data_lines = [l.strip() for l in orig_data_lines]
        #print(f"INFO: data_{example_datapoint} -- # sentences: {len(x)} -- data file: {orig_data_file}")
        
        # Perform inferencing, check to see if this datapoint is correct
        with torch.no_grad():
            y_hat = model.forward(x.reshape([1, len(x), -1]), [kg])[0]
        # Get solutions
        def top_idxs_ordered_desc(input: torch.Tensor):
            return [y for y in sorted([(x, i) for i, x in enumerate(input)])[::-1]]
        max_idx = torch.argmax(y)
        max_idx_hat = torch.argmax(y_hat)
        #print("max index y:", max_idx)
        ordered_desc_idxs = top_idxs_ordered_desc(y_hat)
        #print("ordered desc indices y:", ordered_desc_idxs)
        #print("len desc indices y:", len(ordered_desc_idxs))
        matched = max_idx == max_idx_hat
        #print(f"Is correct?: {matched}")
        if not matched:
            example_datapoint += 1
            print(f"FAILURE: retrying with datapoint {example_datapoint}")

    print(f"INFO: Using datapoint {example_datapoint}. {len(orig_data_lines)} sentences. {orig_data_file}")
    if ordered_desc_idxs[0][0] > 0.98:
        example_datapoint += 1
        continue
    found = True
    print()
    print("Correct sentence:")
    print(f"s{ordered_desc_idxs[0][1]+1:3d}@{ordered_desc_idxs[0][0]:2.3f}: {orig_data_lines[ordered_desc_idxs[0][1]+1]}")
    print()
    print("Next best sentences:")
    N = 3
    i = 1
    found = 0
    while found < N:
        sentence_idx = ordered_desc_idxs[i][1]+1 # +1 because the first sentence is labels
        sentence_prob = ordered_desc_idxs[i][0]
        if sentence_idx >= len(orig_data_lines):
            i += 1
            continue
        print(f"s{sentence_idx:3d}@{sentence_prob:2.4f}: {orig_data_lines[sentence_idx]}")
        i += 1
        found += 1

FAILURE: retrying with datapoint 322
INFO: Using datapoint 322. 114 sentences. synthetic_kaggle_353_973_continuity4.txt
FAILURE: retrying with datapoint 324
FAILURE: retrying with datapoint 325
FAILURE: retrying with datapoint 326
INFO: Using datapoint 326. 51 sentences. synthetic_kaggle_2671_438_continuity3.txt
FAILURE: retrying with datapoint 328
FAILURE: retrying with datapoint 329
FAILURE: retrying with datapoint 330
FAILURE: retrying with datapoint 331
FAILURE: retrying with datapoint 332
FAILURE: retrying with datapoint 333
FAILURE: retrying with datapoint 334
FAILURE: retrying with datapoint 335
FAILURE: retrying with datapoint 336
FAILURE: retrying with datapoint 337
FAILURE: retrying with datapoint 338
FAILURE: retrying with datapoint 339
INFO: Using datapoint 339. 52 sentences. synthetic_kaggle_2429_538_continuity2.txt
INFO: Using datapoint 340. 100 sentences. synthetic_kaggle_2353_843_continuity7.txt
FAILURE: retrying with datapoint 342
FAILURE: retrying with datapoint 343
F

In [16]:
y_hat

tensor([7.3000e-23, 7.3002e-23, 7.3000e-23, 7.3000e-23, 7.3000e-23, 7.3000e-23,
        7.2999e-23, 7.3002e-23, 7.3000e-23, 7.3003e-23, 7.3002e-23, 3.9436e-01,
        7.3001e-23, 7.3007e-23, 6.0564e-01, 7.2998e-23, 7.3000e-23, 7.3001e-23,
        7.3000e-23, 7.3000e-23, 7.3002e-23, 7.3001e-23, 7.3000e-23, 7.3005e-23,
        7.3000e-23, 7.3000e-23, 7.3002e-23, 7.3000e-23, 7.3005e-23, 7.3000e-23,
        7.3005e-23, 7.3000e-23, 4.6077e-19, 7.3000e-23, 7.3001e-23, 7.3002e-23,
        7.3001e-23, 7.3007e-23, 7.3007e-23, 7.3001e-23, 7.3000e-23, 7.3000e-23,
        7.3000e-23, 7.3000e-23, 7.3002e-23, 7.3000e-23, 7.3000e-23, 7.3000e-23,
        7.3001e-23, 6.6559e-19, 7.3000e-23, 7.3000e-23, 7.3000e-23, 7.3000e-23,
        7.3000e-23, 1.4553e-22, 7.3001e-23, 3.9470e-19, 7.3000e-23, 7.3000e-23,
        7.3000e-23, 7.3000e-23, 7.3002e-23, 7.3000e-23, 7.3000e-23, 7.3000e-23,
        7.3000e-23, 7.3000e-23, 2.1239e-22, 7.3000e-23, 7.3005e-23, 7.3002e-23,
        7.3000e-23, 7.3000e-23, 7.3000e-